# Add xAAEnet Blocks to any User Encoder

> Add the xAAEnet blocks needed to explain a user-provided model.

In [ ]:
#| default_exp user_encoder

The goal of this module is to start from any user's own encoder and add the xAAEnet blocks defined previously. We do not replace the user's model idea; we add the components needed to analyze and explain it:

1. keep the user encoder as the feature extractor;
2. project encoder features into the latent space `z`;
3. add a label head from `z`;
4. add the latent discriminator that regularizes `z`;
5. build a symmetric U-Net decoder with `DynamicUnetSkipDropout` from the user encoder and its skip connections.

The encoder must therefore expose spatial feature maps that can be used by the U-Net hooks. For transformer models, the user should provide a wrapper that exposes a spatial feature-map encoder before adding these xAAEnet blocks.

In [ ]:
#| export
from collections.abc import Callable

import torch
import torch.nn as nn
import torch.nn.functional as F
from fastai.torch_core import TensorBase
from torch import Tensor
from pytorch_msssim import ms_ssim

from tell_me_why.model_aae import DynamicUnetSkipDropout


def set_module_trainable(module: nn.Module, trainable: bool) -> nn.Module:
    """Enable or disable gradient updates for every parameter in a module."""
    for parameter in module.parameters():
        parameter.requires_grad = trainable
    return module


def _module_device(module: nn.Module) -> torch.device:
    try:
        return next(module.parameters()).device
    except StopIteration:
        return torch.device("cpu")


class EncoderWithAAEBlocks(nn.Module):
    """User encoder extended with xAAEnet analysis blocks.

    The user encoder stays the feature extractor. The added blocks project its
    features into `z`, predict from `z`, regularize `z`, and reconstruct the
    input with a symmetric U-Net decoder.
    """

    def __init__(
        self,
        encoder: nn.Module,
        input_size: int = 256,
        input_channels: int = 3,
        encoding_dims: int = 128,
        classes: int = 2,
        linear: nn.Module | None = None,
        gen_train: bool = True,
        skip_dropout: float = 1.0,
        freeze_encoder: bool = False,
    ):
        super().__init__()
        assert classes == 2, (
            "tell_me_why supports binary image classification only; classes must be 2."
        )
        self.gen_train = gen_train
        self.input_size = input_size
        self.input_channels = input_channels
        self.encoding_dims = encoding_dims
        self.classes = classes

        if freeze_encoder:
            set_module_trainable(encoder, False)

        self.unet = DynamicUnetSkipDropout(
            encoder=encoder,
            n_out=input_channels,
            img_size=(input_size, input_size),
            skip_dropout=skip_dropout,
            last_cross=False,
        )

        self.flatten = nn.Flatten()
        self.encoder_feature_shape = self._infer_encoder_feature_shape()
        flat_features = int(torch.tensor(self.encoder_feature_shape).prod().item())

        self.fc_encode = nn.Linear(flat_features, encoding_dims)
        self.bn_lin = nn.BatchNorm1d(num_features=encoding_dims)
        self.decoder_fc = nn.Linear(encoding_dims, flat_features)
        self.linear = linear if linear is not None else nn.Linear(encoding_dims, self.classes)

        self.fc_crit1 = nn.Linear(encoding_dims, 64)
        self.fc_crit2 = nn.Linear(64, 16)
        self.fc_crit3 = nn.Linear(16, 1)
        self.bn_crit1 = nn.BatchNorm1d(num_features=64)
        self.bn_crit2 = nn.BatchNorm1d(num_features=16)

    def _infer_encoder_feature_shape(self) -> tuple[int, ...]:
        encoder = self.unet.layers[0]
        was_training = encoder.training
        device = _module_device(encoder)
        try:
            encoder.eval()
            with torch.no_grad():
                sample = torch.zeros(1, self.input_channels, self.input_size, self.input_size, device=device)
                features = encoder(sample)
        finally:
            encoder.train(was_training)
        return tuple(features.shape[1:])

    def latent_gan(self, zi: Tensor) -> Tensor:
        # Same discriminator head as `AAE`: no batch norm on hidden layers.
        x = F.leaky_relu(self.fc_crit1(zi), negative_slope=0.2)
        x = F.leaky_relu(self.fc_crit2(x), negative_slope=0.2)
        return torch.sigmoid(self.fc_crit3(x))

    def reconstruction_loss(self, loss_func: Callable[[Tensor, Tensor], Tensor]) -> Tensor:
        """Optional hook for custom reconstruction objectives (not used by the built-in AAE-style losses)."""
        return loss_func(self.decoder_output, self.input_image)

    def aae_loss_func(self, output: Tensor, target: Tensor) -> Tensor:
        alpha = 0.84

        adversarial_loss = nn.BCELoss()
        if self.gen_train:
            valid = torch.ones_like(self.gan_fake, requires_grad=False).detach()
            self.adv_loss = adversarial_loss(self.gan_fake, valid)
            self.crit_loss = 0
        else:
            valid = torch.ones_like(self.gan_real, requires_grad=False).detach()
            fake = torch.zeros_like(self.gan_fake, requires_grad=False).detach()
            self.real_loss = adversarial_loss(self.gan_real, valid)
            self.fake_loss = adversarial_loss(self.gan_fake, fake)
            self.adv_loss = 0.6 * self.real_loss + 0.4 * self.fake_loss
            self.crit_loss = self.adv_loss

        loss = self.adv_loss

        return loss

    def denoising_ae_loss_func(
        self, clean_xb: Tensor, RECONS_WEIGHT: float, ADV_WEIGHT: float, pred: Tensor, yb: Tensor
    ) -> Tensor:
        # pred and yb are ignored (fastai-style signature); same weighting as `AAE.denoising_ae_loss_func`.
        alpha = 0.84
        l1_loss = F.l1_loss(self.decoder_output, clean_xb)
        ms_ssim_val = ms_ssim(self.decoder_output, clean_xb, data_range=1.0, size_average=True)
        msssim_loss = 1.0 - ms_ssim_val
        self.recons_loss = alpha * msssim_loss + (1.0 - alpha) * l1_loss
        adversarial_loss = nn.BCELoss()
        if self.gen_train:
            valid = torch.ones_like(self.gan_fake, requires_grad=False).detach()
            self.adv_loss = adversarial_loss(self.gan_fake, valid)
            self.crit_loss = 0
        else:
            valid = torch.ones_like(self.gan_real, requires_grad=False).detach()
            fake = torch.zeros_like(self.gan_fake, requires_grad=False).detach()
            self.real_loss = adversarial_loss(self.gan_real, valid)
            self.fake_loss = adversarial_loss(self.gan_fake, fake)
            self.adv_loss = 0.6 * self.real_loss + 0.4 * self.fake_loss
            self.crit_loss = self.adv_loss
        self.recons_loss = RECONS_WEIGHT * self.recons_loss + ADV_WEIGHT * self.adv_loss
        return self.recons_loss

    def classif_loss_func(
        self, output: Tensor, target: Tensor, ADV_WEIGHT: float, RECONS_WEIGHT: float, CLASS_WEIGHT: float, **kwargs
    ) -> Tensor:
        alpha = 0.84
        l1_loss = F.l1_loss(self.decoder_output, self.input_image)
        ms_ssim_val = ms_ssim(self.decoder_output, self.input_image, data_range=1.0, size_average=True)
        msssim_loss = 1.0 - ms_ssim_val
        self.recons_loss = alpha * msssim_loss + (1.0 - alpha) * l1_loss
        self.classif_loss = F.cross_entropy(output, target, **kwargs)
        adversarial_loss = nn.BCELoss()
        if self.gen_train:
            valid = torch.ones_like(self.gan_fake, requires_grad=False).detach()
            self.adv_loss = adversarial_loss(self.gan_fake, valid)
            self.crit_loss = 0
        else:
            valid = torch.ones_like(self.gan_real, requires_grad=False).detach()
            fake = torch.zeros_like(self.gan_fake, requires_grad=False).detach()
            self.real_loss = adversarial_loss(self.gan_real, valid)
            self.fake_loss = adversarial_loss(self.gan_fake, fake)
            self.adv_loss = 0.6 * self.real_loss + 0.4 * self.fake_loss
            self.crit_loss = self.adv_loss

        loss = ADV_WEIGHT * self.adv_loss + RECONS_WEIGHT * self.recons_loss + CLASS_WEIGHT * self.classif_loss
        return loss

    def pure_classif_loss_func(self, pred: Tensor, target: Tensor, **kwargs) -> Tensor:
        return F.cross_entropy(pred, target, **kwargs)

    def forward(self, x: Tensor) -> Tensor:
        self.input_image = x

        feats = self.unet.layers[0](x)
        flat = self.flatten(feats)
        self.z = self.fc_encode(flat)

        labels = self.linear(self.z)
        self.gan_fake = self.latent_gan(self.z)
        self.gan_real = self.latent_gan(torch.randn_like(self.z))

        z_spatial = F.relu(self.decoder_fc(self.z))
        z_spatial = z_spatial.view(-1, *self.encoder_feature_shape)
        out = TensorBase(z_spatial)
        orig_x = TensorBase(torch.zeros_like(self.input_image))

        for layer in self.unet.layers[1:]:
            out.orig = orig_x
            nres = layer(out)
            out.orig = None
            if hasattr(nres, "orig"):
                nres.orig = None
            out = nres

        self.decoder_output = out
        return labels


def add_xaae_blocks(
    encoder: nn.Module,
    input_size: int = 256,
    input_channels: int = 3,
    encoding_dims: int = 128,
    classes: int = 2,
    linear: nn.Module | None = None,
    gen_train: bool = True,
    skip_dropout: float = 1.0,
    freeze_encoder: bool = False,
) -> EncoderWithAAEBlocks:
    """Add xAAEnet analysis blocks to a user encoder."""
    return EncoderWithAAEBlocks(
        encoder=encoder,
        input_size=input_size,
        input_channels=input_channels,
        encoding_dims=encoding_dims,
        classes=classes,
        linear=linear,
        gen_train=gen_train,
        skip_dropout=skip_dropout,
        freeze_encoder=freeze_encoder,
    )

## Practical Use Case

The user provides the encoder. `add_xaae_blocks` adds the xAAEnet analysis blocks and automatically builds the matching U-Net decoder through `DynamicUnetSkipDropout`.

The forward pass returns `labels`.

In [ ]:
user_encoder = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
)

model = add_xaae_blocks(
    encoder=user_encoder,
    input_size=256,
    input_channels=3,
    encoding_dims=16,
    classes=2,
)

# MS-SSIM (used in `classif_loss_func` / `denoising_ae_loss_func`) needs spatial size > ~160.
batch = torch.randn(2, 3, 256, 256)
labels = model(batch)
labels.shape, model.z.shape, model.decoder_output.shape

## Training Loss

Defined Losses:

- `denoising_ae_loss_func(clean_xb, RECONS_WEIGHT, ADV_WEIGHT, pred, yb)` is for denoising-style pretraining: reconstruction vs `clean_xb` plus the GAN term; `pred` and `yb` are ignored but kept for a fastai-compatible signature.
- `aae_loss_func(output, target)` returns **only** the adversarial latent loss (same alternating generator / discriminator logic as `AAE`).
- `classif_loss_func(output, target, ADV_WEIGHT, RECONS_WEIGHT, CLASS_WEIGHT, **kwargs)` combines MS-SSIM + L1 reconstruction on `decoder_output` vs `input_image`, the latent GAN term (`gan_fake` / `gan_real`), and cross-entropy on the logits.

The user encoder path still infers `encoder_feature_shape` automatically so the U-Net and bottleneck stay consistent with **your** spatial encoder.

Because reconstruction uses MS-SSIM, training images should be **wider and taller than about 160 pixels**; the example below uses `256×256` so `nbdev` tests and `classif_loss_func` run without assertion errors from `pytorch_msssim`.

In [ ]:
target_labels = torch.tensor([0, 1])

classif_loss = model.classif_loss_func(
    labels, target_labels, ADV_WEIGHT=0.1, RECONS_WEIGHT=0.1, CLASS_WEIGHT=1.0
)

aae_loss = model.aae_loss_func(labels, target_labels)

classif_loss.shape, aae_loss.shape